# DADA2: 04 — Evaluation

Three TODOs.

You will conclude the assignment by running a RAG eval.

Three questions to answer with numbers:
- Is the decision model as good a router as the chat model? 
- Does the diagnosis agent beat the ladder, and at what cost?
- which parts of what you built were worth building?

In [1]:
from dotenv import load_dotenv
load_dotenv()

import time
from rag import config, decide, evaluate, retrieval, agent, ladder

## Read this first

This notebook is where the Gemini quota goes. The ladder agent spends about
five chat calls per question and yours about three, and the judge spends one
more on each answer. Fourteen questions through both agents is roughly 150
calls against a limit of 500 a day.

So, you will want to run it once, when notebooks 02 and 03 already pass their final cells. The comparison cells sleep between questions on purpose. Start them and leave them alone.

In [2]:
specific = evaluate.load_evalset("specific")
comparative = evaluate.load_evalset("comparative")
items = specific + comparative

print(f"{len(specific)} specific + {len(comparative)} comparative = {len(items)} labeled questions")
print("reranker:       cohere")
print("chat model:    ", config.CHAT_MODEL)
print("decision model:", config.DECISION_MODEL)

14 specific + 10 comparative = 24 labeled questions
reranker:       cohere
chat model:     gemini-3.5-flash-lite
decision model: clef-flash


## 1. Two routers

### TODO 4.1 — the decision model against the chat model (1 pt)

Every labeled question has a kind: the 14 in `evalset.json` are specific,
the 10 in `evalset_hard.json` are comparative. Ask both models which kind each
one is, and keep the time.

Write the chat-model version yourself. It is a `with_structured_output` call
with a `Literal` field, the same thing you did for `Filter` in notebook 02.
Use `decide.SCOPES` for the descriptions so both models are asked the same
question.

**Steps:**

1. Define a scoper that uses Scope to get structured output from chat model
3. In the loop below, the decision-model line is written. Add the same two lines for the chat model, keyed `"chat"`: time the call, add the seconds, add one to `right` when the choice matches `kind`.


In [3]:
from typing import Literal
from pydantic import BaseModel, Field
from rag.util import retry

# TODO 4.1 — a scoper that uses Scope to get structured output from chat model
class Scope(BaseModel):
    """Which kind of question this is."""
    scope: Literal["specific", "comparative", "figure"] = Field(
        description="; ".join(f"{k}: {v}" for k, v in decide.SCOPES.items()))

def chat_scope(question):
    t = time.time()
    out = retry(scoper.invoke, "Question about SEC 10-K filings: " + question
                + "\nWhat does it ask for?")
    return out.scope, time.time() - t

scoper = config.chat().with_structured_output(Scope)  # TODO 4.1
# your code here

In [4]:
labeled = [(i["question"], "specific") for i in specific] + \
           [(i["question"], "comparative") for i in comparative]

right = {"decision": 0, "chat": 0}
secs = {"decision": 0.0, "chat": 0.0}
for q, kind in labeled:
    t = time.time(); choice, _ = decide.scope(q); secs["decision"] += time.time() - t
    right["decision"] += (choice == kind)
    t = time.time(); choice, _ = chat_scope(q); secs["chat"] += time.time() - t
    right["chat"] += (choice == kind)
    time.sleep(3)

n = len(labeled)
for name in ("decision", "chat"):
    print(f"{name:9} {right[name]}/{n} right, {secs[name]/n:.2f}s per call")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


decision  20/24 right, 0.49s per call
chat      18/24 right, 0.82s per call


NOTE: Accuracy is close, and the chat model is probably not
slower. If you expected the decision model to win on both, write down what it does give you instead: a probability per option, and a call that does not count against the daily limit. Whether those are worth having is the question
the next section answers.

## 2. Two agents

### TODO 4.2 — the ladder against the diagnosis (1 pt)

`evaluate.compare_agents` runs each agent over the questions, has the chat
model judge every answer, and keeps the bill: chat calls, decision calls, and
seconds per question. Both agents expose `run(question)` returning the final
state, which is all it needs.

Fourteen questions is enough. Use eight specific and six comparative so both
kinds are in the mix.

**Steps:** nothing to write. Run the cell once, when both agents pass their final cells. If a 429 cuts it off, score the two agents in separate cells rather than restarting the pair.

This will take a while. Plan accordingly. 

Make sure your rate limit is reset or use a different provider (e.g., Groq, OpenRouter) instead of the Gemini API. 

Let it run and come back to see your results.


In [7]:
# TODO 4.2
subset = specific[:8] + comparative[:6]

rows = evaluate.compare_agents({"ladder": ladder.run, "diagnosis": agent.run}, subset)

  agent         accuracy  chat/q  decision/q    s/q
  --------------------------------------------------


/Users/karthikeyansura/Downloads/CCIS Playground/CS6983-AAI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ladder            0.79     5.1         0.0   21.1
  diagnosis         0.86     3.1         2.5   16.0


In [9]:
# the judge's reasons, on anything either agent got wrong
_, notes = evaluate.score_agent(agent.run, subset[:4])
for note in notes:
    mark = "ok " if note["correct"] else "WRONG"
    print(f"[{mark}] {note['question'][:66]}  ({note['chat_calls']} chat, {note['decision_calls']} decision)")
    print(f"        {note['reason'][:150]}\n")

[ok ] What does MU state that the preparation of financial statements an  (2 chat, 2 decision)
        CORRECT
The provided answer accurately identifies management's responsibility to make estimates and judgments for financial statement preparation unde

[ok ] What termination fee is AMD required to pay ZT Systems under certa  (2 chat, 2 decision)
        CORRECT
The answer correctly identifies the $300 million termination fee related to the failure to obtain regulatory approvals as stated in the AMD fi

[ok ] What factors does NVIDIA believe are the principal competitive fac  (2 chat, 2 decision)
        CORRECT
The provided answer accurately lists the principal competitive factors for NVIDIA for fiscal year 2025 as stated in their SEC filings.

[WRONG] What competitive pressures from Intel and Nvidia did AMD face in f  (2 chat, 2 decision)
        INCORRECT: The provided text refers to the future year 2025, which would not be the current reporting period for an SEC filing analyzed in

If the diagnosis agent got something wrong that the ladder got right, find
out which half failed. Run `decide.scope` and `decide.diagnose` on that
question by hand and look at the probabilities. A wrong route and a wrong
verdict are different bugs, and the fix for one is a threshold while the fix
for the other is a search strategy.

### TODO 4.3 — write it up (2 pts)

In the cell below, in your own words. Four or five short paragraphs.

**Conditions.** Reranker, chat model, decision model, k, chunk size. Enough
for someone to re-run you.

**The routers.** Which was more accurate, which was faster, and by how much.
Then say what the decision model is for, given those numbers.

**The agents.** Accuracy, chat calls per question, seconds per question, for
both. Name the question where they differed most and say why.

**One thing that did not help.** Something you built that the numbers say you
could remove. Say what the mechanism is, not that the model is better at it.

**What you would ship**, and what you would want to know about your users'
questions first.

Two things that cost marks: an improvement the table does not show, and a
difference explained by "the model is better" when the reason is in your
code.

**Steps:** write it as prose, four or five short paragraphs. The bold headings in the cell below are the order; replace the cell's text under each. Every number you quote should be in a table above.


---

### Your write-up

**Conditions.** The chat model and the judge are `gemini-3.5-flash-lite`, and the decision model is `clef-flash`. The embeddings are `@cf/qwen/qwen3-embedding-0.6b` on Cloudflare (1024 dimensions), in the Pinecone index `dada2-1024`. The reranker is Cohere `rerank-v3.5`. The 27 section files make 2,124 chunks of up to 1,200 characters, with an overlap of 150. Retrieval uses k=5, a rerank pool of 10 and an RRF constant of 60. The router test uses all 24 labeled questions. The agent test uses the first 8 specific and the first 6 comparative questions, with one run of each agent.

**The routers.** The decision model was right on 20 of 24 questions, at 0.49 s per call. The chat model was right on 18 of 24, at 0.82 s per call. The decision model also gives a probability for each option, and its calls do not use the Gemini quota. Below `SURE` (0.6), the agent starts with `filter+rrf`. One limit is in the code: `after_route` reads the label, not the probability, so an unsure "figure" label still goes to `lookup`.

**The agents.** The judge scored the ladder 0.79 and the diagnosis agent 0.86 on 14 questions, which is one more correct answer. The diagnosis agent used 3.1 chat calls per question against 5.1, and 16.0 s against 21.1 s. Rate-limit retries can add time. The notebook 03 traces show how the paths differ. On the TSMC and dress-code questions, the ladder climbed all three rungs for 9 chat calls each. The diagnosis agent planned one more search for TSMC (4 chat calls in the module check) and stopped dress code at `absent` (0.90). The judge sees only the question and the answer. In notebook 04 it marked an AMD answer WRONG because it took 2025 to be a future year.

**What did not help.** On six specific questions at k=5, `filter` had the best MRR (0.917). `filter+rrf` had 0.833, `filter+rrf+rerank` 0.889 and `hybrid+rrf` 0.806, all at recall@5 1.00. The filter hurt five comparative questions at k=10: recall fell from 0.83 to 0.57. BM25 on "HBM3E" returned five chunks from `AMD_2023_business.txt`, which does not look like an exact-term match. In the code, `local_chunks` collapses whitespace and Pinecone keeps the raw text, so RRF may not merge a chunk pair whose copies differ in whitespace. I did not measure this. These are small samples, not general rules.

**What I would ship.** I would ship the diagnosis agent: one more judged-correct answer, fewer chat calls and less time per question. I would also give the judge the passages and the reference answer, and check whether short heading chunks crowd out longer passages. I have not tested either change. First, I want to know how many user questions name one company and one year, and how many compare companies, because that decides the best first search.

Agent comparison as the judge recorded it (notebook 04, TODO 4.2). 14 questions, k=5.

| agent | accuracy | chat/q | decision/q | s/q |
|---|---|---|---|---|
| ladder | 0.79 | 5.1 | 0.0 | 21.1 |
| diagnosis | 0.86 | 3.1 | 2.5 | 16.0 |

**Copy your write-up to README.md.**